In [2]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd

In [3]:
project_root = os.path.dirname(os.path.dirname("test.ipynb"))
data_dir = os.path.join(project_root, "data")
raster_dir = os.path.join(data_dir, "rasters")
transient_dir = os.path.join(project_root, "transients")
output_dir = os.path.join(project_root, "outputs")

energy_raster_path = os.path.join(raster_dir, "GHS_BUILT_S_timeseries_points.gpkg")


shapefile_path = os.path.join(data_dir, "ne_10m_admin_0_countries.shp")  # may need other files rather than just shp?
raster_path = os.path.join(data_dir, "gpw_v4_population_density_rev11_2020_30_min.tif")
country_energy_path = os.path.join(data_dir, "Country Energy Data.xlsx")

In [56]:
energy_timeseries = gpd.read_file(energy_raster_path)
countries = set(energy_timeseries["country"])

year = 2020
tot_leds = 3500
place_ocean = True
all_leds_gdf = gpd.GeoDataFrame()

In [57]:
led_data = pd.read_csv("data/API/global_energy_consumption.csv")
led_data = led_data.pivot(index="Entity", columns="Year", values="primary_energy_consumption__twh").reset_index()
led_data = led_data[led_data["Entity"].isin(countries)]

missing = max(led_data.isna().sum())
idx = (np.abs(led_data.columns.values[1:-1] - year)).argmin() + 1
energy_year = led_data.columns.values[idx]
while missing > 10:
    missing = led_data.isna().sum().loc[energy_year]
    if missing > 10:
        energy_year -= 1

In [58]:
led_data = led_data[['Entity', energy_year]]
total_energy = led_data[energy_year].sum()
led_data['energy_prop'] = led_data[energy_year] / total_energy
led_data['num_leds'] = led_data['energy_prop'] * tot_leds


In [59]:
def rounding(led_data):
    return np.round(led_data["num_leds"]).astype(int) if led_data["num_leds"] > 1 else np.floor(led_data["num_leds"]).astype(int)

led_data["Round"] = led_data.apply(rounding, axis=1)
led_data

Year,Entity,2020,energy_prop,num_leds,Round
0,Afghanistan,46.911743,3.555890e-04,1.244562,1
4,Albania,21.116934,1.600655e-04,0.560229,0
5,Algeria,648.898800,4.918625e-03,17.215188,17
7,Angola,76.321830,5.785162e-04,2.024807,2
8,Antarctica,0.046016,3.487985e-07,0.001221,0
...,...,...,...,...,...
265,Venezuela,527.860170,4.001158e-03,14.004051,14
266,Vietnam,1228.728300,9.313708e-03,32.597978,33
273,Yemen,33.195854,2.516232e-04,0.880681,0
275,Zambia,38.494797,2.917889e-04,1.021261,1


In [60]:
for index, row in led_data.iterrows():

    country_name = row['Entity']
    num_leds = int(row['Round'])
    leds_placed = 0

    values_array = energy_timeseries[energy_timeseries['country'] == country_name]
    values_array = values_array[["point_index", f"{year}", "geometry"]].sort_values(f"{year}", ascending=False)

    available_cells = len(values_array)
    missing_leds = num_leds - available_cells
    
    if available_cells == 0:
        print(f"Could not find {country_name} in raster data, skipping...")

    else:

        for leds in range(0,num_leds-leds_placed): # Place LEDs on the land-space
            
            if leds_placed < available_cells: 
                
                all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_array["geometry"].iloc[leds]],
                                                                        'Country': [country_name],
                                                                        'Raster_Density': [values_array[f"{year}"].iloc[leds]]
                                                                        }, geometry='geometry')], ignore_index=True)
                leds_placed += 1


            else:

                if place_ocean == True:
                    
                    if leds_placed >= num_leds:
                        break

                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {available_cells} cells are available. Attempting to place {missing_leds} LEDs in surrounding area.")
                    values_sorted = energy_timeseries.sort_values('point_index').reset_index(drop=True)
                    surround = 1
                    
                    while leds_placed < num_leds:

                        surround_indices = ([x - (720*surround) for x in values_array.point_index] + 
                                            [x + (720*surround) for x in values_array.point_index] +
                                            [x - surround for x in values_array.point_index] + 
                                            [x + surround for x in values_array.point_index]) 
                        surround_indices = np.unique(list(filter(lambda x: x >= 0, surround_indices)))
                        values_filtered = energy_timeseries[energy_timeseries["point_index"].isin(surround_indices)].query("country.isnull()")

                        for leds in range(len(values_filtered)): # Place remaining LEDs on the land-space
                            all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_filtered["geometry"].iloc[leds]],
                                                                                    'Country': [country_name],
                                                                                    'Raster_Density': [values_filtered[f"{year}"].iloc[leds]]
                                                                                    }, geometry='geometry')], ignore_index=True)
                            leds_placed += 1
                            if leds_placed >= num_leds:
                                break
                        surround += 1
                
                else: 
                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {leds_placed} were placed due to not having enough space.")
                    break

In [63]:
all_leds_gdf.to_file("dataframe.gpkg", driver="GPKG")

c:\ProgramData\miniforge3\envs\science_gen\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
